# **Notebook 3 - Basket Analysis**

Afonso Fernandes 20241710, Lourenço Lima 20241711, Lucas Casimiro 20241796

## Imports

In [ ]:
import os
import sys
import warnings
from pathlib import Path

def _find_project_root(start, marker="requirements.txt"):
    path = Path(start).resolve()
    for candidate in [path] + list(path.parents):
        if (candidate / marker).exists():
            return str(candidate)
    raise RuntimeError(f"Could not find project root (marker={marker!r}, searched from {start})")

PROJECT_ROOT = _find_project_root(os.path.abspath("."))
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import seaborn as sns
import math
from sklearn.preprocessing import RobustScaler


from functions.eda import *
from functions.preprocessing import *
from functions.basket import *
%load_ext autoreload
%autoreload 2

# **1. Merging data**

In [ ]:
customer_basket = pd.read_csv('data/customer_basket.csv')
customer = pd.read_csv('data/ci_clustered.csv')

In [ ]:
customer_merged = pd.merge(
    customer_basket, 
    customer[['customer_id', 'final_cluster_label']], 
    on='customer_id', 
    how='inner')
customer_merged

In [ ]:
customer_merged_grouped = customer_merged.groupby('final_cluster_label')
customer_merged_grouped
cluster_dataframes = {cluster: group for cluster, group in customer_merged_grouped}

In [ ]:
loyal_core_spenders = cluster_dataframes.get('Loyal core spenders')
bargain_hunters = cluster_dataframes.get('Bargain hunters')
big_families = cluster_dataframes.get('Big families (big spenders)')
vegans = cluster_dataframes.get('Vegans')
tech_enthusiasts = cluster_dataframes.get('Tech enthusiasts')
karens = cluster_dataframes.get('Karens')
gamers= cluster_dataframes.get('Gamers')

# Separete each cluster in it's dataframe

In [ ]:
cluster_dataframes = {
    cluster: group.reset_index(drop=True)
    for cluster, group in customer_merged.groupby('final_cluster_label')
}

cluster_dataframes.keys()

## Create the variables

In [ ]:
loyal_core_spenders = cluster_dataframes.get('Loyal core spenders')
bargain_hunters = cluster_dataframes.get('Bargain hunters')
big_families = cluster_dataframes.get('Big families (big spenders)')
vegans = cluster_dataframes.get('Vegans')
tech_enthusiasts = cluster_dataframes.get('Tech enthusiasts')
karens = cluster_dataframes.get('Karens')
gamers= cluster_dataframes.get('Gamers')

# Association Rules

In [ ]:
for cluster_name, cluster_df in cluster_dataframes.items():
    print(f"\n===== Cluster {cluster_name} =====")

    rules = generate_association_rules_from_cluster(
        cluster_df,
        min_support=0.02,
        metric='lift',
        min_threshold=1,
        only_simple_rules=True
    )

    if rules.empty:
        print("No rules found.")
    else:
        display(rules.head(20))

Note: some baskets were lost due to outlier removal.


**Temos q ver como fazemos com isto, se so aceitamos ou corremos os notebooks todos com os outliers mm**